# 34 · Load: BLS JOLTS and LAUS, New York → `staging`

**Source cards:** `bls_jolts.md`, `bls_laus.md`. Context series for the labor market around the job projections: how many openings, hires and quits (JOLTS), and how many people are working or looking (LAUS).

| Table | One row = | Expected rows |
|---|---|---|
| `stg_bls__jolts` | JOLTS series × year × period, U.S. and New York State, total nonfarm, 2015+ | 6,793 |
| `stg_bls__laus` | LAUS series × year × period, NYC, its 5 counties, NY metro, NY State, rest of state, 2015+ | 8,582 |

BLS bulk time-series files (snapshot 2026-10-12). JOLTS is a survey of about 21,000 employers; state estimates are model-based, total nonfarm only, levels in thousands. LAUS is BLS/state modeled estimates from the household survey, payroll data and UI claims; counties and NYC are not seasonally adjusted. October 2025 was not collected (federal shutdown): the files show `-` with footnote `X` (LAUS) or `9` (JOLTS), staged as NULL with the footnote kept. `P` = preliminary.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 80, "display.width", 220)
NB = "34-load-bls-jolts-laus.ipynb"
FIRST_YEAR = 2015
conn = connect()

def reader(key):
    F = raw_files(key)
    def rd(stem: str):
        row = F[F.file.str.contains(f"/{stem}_")].iloc[-1]
        d = pd.read_csv(row.path, sep="\t", dtype=str, keep_default_na=False)
        return d.rename(columns=str.strip).apply(lambda s: s.str.strip()), row.file
    return rd

def parse(c):
    """Shared BLS period parsing: numbers (dash -> NULL), month / annual average, footnote."""
    assert c.period.str.fullmatch(r"M(0[1-9]|1[0-3])").all(), sorted(set(c.period))
    v = pd.to_numeric(c.value, errors="coerce")
    print("  non-numeric values -> NULL:", c.value[v.isna()].value_counts().to_dict() or "none",
          "| their footnotes:", c.footnote_codes[v.isna()].value_counts().to_dict())
    c["value"] = v.astype("float64")
    c["year"] = c.year.astype(int).astype("Int64")
    c["period_type"] = c.period.map(lambda p: "annual_avg" if p == "M13" else "month")
    c["month"] = c.period.str[1:].astype(int).where(c.period != "M13").astype("Int64")
    c["footnote_code"] = c.footnote_codes.replace("", None)
    return c

## 1. JOLTS: U.S. and New York State, total nonfarm, all sizes

In [2]:
rd = reader("bls_jt")
jd, f_jd = rd("jt-data-1-AllItems")
js, _ = rd("jt-series")
el, _ = rd("jt-dataelement")
rl, _ = rd("jt-ratelevel")
keep = js[js.state_code.isin(["00", "36"]) & (js.industry_code == "000000") & (js.sizeclass_code == "00") & (js.area_code == "00000")]
print(f"series kept: {len(keep)} ({keep.groupby(['state_code', 'seasonal']).size().to_dict()})")
j = jd[jd.series_id.isin(keep.series_id) & (jd.year.astype(int) >= FIRST_YEAR)].copy()
log(f"keep U.S. + NY total nonfarm series, {FIRST_YEAR}+", len(jd), len(j))
assert not j.duplicated(["series_id", "year", "period"]).any()
j = parse(j).merge(keep[["series_id", "state_code", "seasonal", "dataelement_code", "ratelevel_code"]], on="series_id")
j = j.merge(el[["dataelement_code", "dataelement_text"]], on="dataelement_code").merge(rl[["ratelevel_code", "ratelevel_text"]], on="ratelevel_code")
j["area_name"] = j.state_code.map({"00": "United States", "36": "New York"})
assert j.dataelement_text.notna().all() and len(j) == j.series_id.map(lambda s: 1).sum()

# M13 is not always an average. BLS (jt.txt): "Annual estimates are available ... for hires, quits, layoffs and discharges,
# other separations, and total separations": for these flows, M13 is the 12-month TOTAL. For openings (a stock) and all rates it is the average.
flow = (j.ratelevel_code == "L") & j.dataelement_code.isin(["HI", "QU", "LD", "OS", "TS"])
j.loc[(j.period == "M13") & flow, "period_type"] = "annual_total"
print("M13 rows by type:", j[j.period == "M13"].period_type.value_counts().to_dict())

# Tie-out: annual total = sum of the 12 months (flows), annual average = mean of the 12 months (openings, rates)
m = j[(j.seasonal == "U") & (j.period_type == "month")].groupby(["series_id", "year"]).value.agg(["sum", "mean", "count"])
a = j[(j.seasonal == "U") & (j.period == "M13")].set_index(["series_id", "year"])[["value", "period_type"]]
t = m.join(a, how="inner").query("count == 12")
t["diff"] = (t["sum"].where(t.period_type == "annual_total", t["mean"]) - t.value).abs()
lv = t.index.get_level_values(0).str.endswith("L")
print(f"annual values checked: {len(t)} | max gap, levels: {t['diff'][lv].max():.2f} thousand | rates: {t['diff'][~lv].max():.2f} pt")
assert t["diff"][lv].max() <= 1 and t["diff"][~lv].max() <= 0.1

ny = j[(j.state_code == "36") & (j.seasonal == "S") & (j.ratelevel_code == "L") & (j.period_type == "month")]
display(ny.pivot_table(index="year", columns="dataelement_text", values="value", aggfunc="mean").loc[2019:].round(0)
        .rename_axis(columns="NY State, monthly average, thousands (SA)"))

series kept: 48 ({('00', 'S'): 13, ('00', 'U'): 14, ('36', 'S'): 11, ('36', 'U'): 10})
  keep U.S. + NY total nonfarm series, 2015+: 627,672 -> 6,793 rows (-620,879)
  non-numeric values -> NULL: {'-': 4} | their footnotes: {'9': 4}
M13 rows by type: {'annual_avg': 143, 'annual_total': 99}
annual values checked: 242 | max gap, levels: 0.50 thousand | rates: 0.08 pt


"NY State, monthly average, thousands (SA)",Hires,Job openings,Layoffs and discharges,Quits,Total separations
year,,,,,
2019,296.0,389.0,114.0,149.0,282.0
2020,302.0,341.0,215.0,124.0,359.0
2021,307.0,548.0,84.0,161.0,267.0
2022,307.0,554.0,82.0,178.0,279.0
2023,284.0,446.0,100.0,156.0,278.0
2024,272.0,467.0,95.0,160.0,273.0
2025,311.0,473.0,125.0,166.0,309.0


## 2. LAUS: NYC, the 5 boroughs, NY metro, NY State, rest of state

In [3]:
rd = reader("bls_la")
ld, f_ld = rd("la-data-39-NewYork")
ls, _ = rd("la-series")
la, _ = rd("la-area")
me, _ = rd("la-measure")
AREAS = {"ST3600000000000": "state", "CT3651000000000": "city", "MT3635620000000": "metro", "BS3600000000000": "balance",
         "CN3600500000000": "county", "CN3604700000000": "county", "CN3606100000000": "county",
         "CN3608100000000": "county", "CN3608500000000": "county"}
lk = ls[ls.area_code.isin(AREAS)]
print(f"series kept: {len(lk)}")
display(lk.groupby(["area_code", "seasonal"]).measure_code.agg(lambda s: ",".join(sorted(s))).unstack())
l = ld[ld.series_id.isin(lk.series_id) & (ld.year.astype(int) >= FIRST_YEAR)].copy()
log(f"keep NYC-area series, {FIRST_YEAR}+", len(ld), len(l))
assert not l.duplicated(["series_id", "year", "period"]).any()
l = parse(l).merge(lk[["series_id", "area_code", "measure_code", "seasonal"]], on="series_id")
l = l.merge(la[["area_code", "area_text"]], on="area_code").merge(me[["measure_code", "measure_text"]], on="measure_code")
l["area_type"] = l.area_code.map(AREAS)
l["county_fips"] = l.area_code.str[2:7].where(l.area_type == "county")
assert l.area_text.notna().all() and l.measure_text.notna().all()

# Tie-out: NYC (not adjusted) = sum of its 5 counties, for labor force, employment, unemployment
k = ["year", "period", "measure_code"]
cnt = l[(l.area_type == "county") & l.measure_code.isin(["04", "05", "06"])].groupby(k).value.agg(["sum", "count"])
city = l[(l.area_type == "city") & (l.seasonal == "U") & l.measure_code.isin(["04", "05", "06"])].set_index(k).value
t = cnt.join(city.rename("city"), how="inner").query("count == 5").dropna()
t["diff"] = t["sum"] - t.city
print(f"NYC vs sum of 5 counties: {len(t)} month x measure cells | max abs gap {t['diff'].abs().max():,.0f} people")
assert t["diff"].abs().max() <= 5

u = l[(l.area_type.isin(["city", "state"])) & (l.seasonal == "U") & (l.measure_code == "03") & (l.period_type == "annual_avg")]
display(u.pivot_table(index="year", columns="area_text", values="value").loc[2019:].rename_axis(columns="Unemployment rate, annual average (%)"))

series kept: 58


seasonal,S,U
area_code,,
BS3600000000000,"03,04,05,06","03,04,05,06"
CN3600500000000,NaN,"03,04,05,06"
CN3604700000000,NaN,"03,04,05,06"
CN3606100000000,NaN,"03,04,05,06"
CN3608100000000,NaN,"03,04,05,06"
CN3608500000000,NaN,"03,04,05,06"
CT3651000000000,"03,04,05,06,07,08","03,04,05,06,07,08,09"
MT3635620000000,NaN,"03,04,05,06"
ST3600000000000,"03,04,05,06,07,08","03,04,05,06,07,08,09"


  keep NYC-area series, 2015+: 398,996 -> 8,582 rows (-390,414)
  non-numeric values -> NULL: {'-': 58} | their footnotes: {'X': 58}
NYC vs sum of 5 counties: 450 month x measure cells | max abs gap 3 people


"Unemployment rate, annual average (%)",New York,"New York city, NY"
year,,
2019,3.9,4.0
2020,9.8,12.2
2021,7.1,10.1
2022,4.3,5.6
2023,4.0,5.0
2024,4.2,5.0
2025,4.3,5.2


## 3. Stage

In [4]:
cols = ["series_id", "year", "period", "period_type", "month", "value", "footnote_code"]
s_j = (j.assign(source_file=f_jd)[cols[:1] + ["state_code", "area_name", "seasonal", "dataelement_code", "dataelement_text",
                                               "ratelevel_code", "ratelevel_text"] + cols[1:] + ["source_file"]]
       .sort_values(["series_id", "year", "period"]).reset_index(drop=True))
display(stage(s_j, "stg_bls__jolts", ["series_id", "year", "period"], expected_rows=6_793, source_key="bls_jt", notebook=NB, conn=conn,
              comment="BLS JOLTS, U.S. (00) and New York State (36), total nonfarm, all size classes, SA and NSA, 2015+. Levels in thousands; rates in %. M13 = annual total for hire/separation levels, annual average otherwise (see period_type)."))

s_l = (l.assign(source_file=f_ld)[cols[:1] + ["area_code", "area_type", "area_text", "county_fips", "measure_code", "measure_text", "seasonal"]
                                  + cols[1:] + ["source_file"]]
       .sort_values(["series_id", "year", "period"]).reset_index(drop=True))
display(stage(s_l, "stg_bls__laus", ["series_id", "year", "period"], expected_rows=8_582, source_key="bls_la", notebook=NB, conn=conn,
              comment="BLS LAUS, New York City, its 5 counties, NY-NJ metro (35620), New York State, balance of state, 2015+. People or %. M13 = annual average."))
if conn is not None:
    conn.close()

  stg_bls__jolts: 6,793 rows x 15 cols, key ['series_id', 'year', 'period'] unique | local data/interim/staging/stg_bls__jolts.parquet | DB: 6793


,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_bls__jolts,6793,15,series_id + year + period,bls_jt,34-load-bls-jolts-laus.ipynb,6793,2026-10-14T22:10:48+00:00


  stg_bls__laus: 8,582 rows x 15 cols, key ['series_id', 'year', 'period'] unique | local data/interim/staging/stg_bls__laus.parquet | DB: 8582


,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_bls__laus,8582,15,series_id + year + period,bls_la,34-load-bls-jolts-laus.ipynb,8582,2026-10-14T22:10:48+00:00
